# 04 — Layout analysis (PP-StructureV3)

**Phase 4** (plan §6 Stage 4). Runs after `03b_fusion.ipynb`.

Goal: layer **structural understanding** on top of the flat `textFused`
produced in Phase 3. PP-StructureV3 (shipped inside the already-installed
`paddleocr` package — no new dependency) detects layout regions, recovers
correct multi-column reading order, and produces a structured Markdown
rendition of each page.

This phase does **not** replace Phase 3. `textFused` remains the canonical
character-level transcription. `structuredMarkdown` is the structural
wrapper consumed by Phase 5 (chunking).

**Page-type classification** (secondary outcome):
Pages where both OCR engines found ≤10 characters (handwritten cursive
facsimiles, Dunhuang manuscripts) are pre-classified as
`pageType='manuscript_cursive'` and receive `layoutStatus='manuscript'`
without any inference. Phase 5 falls back to `textFused` for these.

**New PAGE properties written:**

| Property | Type | Values |
|---|---|---|
| `layoutStatus` | string | `'ok'` / `'empty'` / `'failed'` / `'manuscript'` |
| `layoutAt` | datetime | `timestamp()` |
| `layoutModelVersion` | string | `'PP-StructureV3/PP-DocLayout_plus-L'` |
| `layoutDurationSeconds` | float | wall-clock |
| `layoutRegionCount` | int | detected bounding boxes |
| `layoutJson` | string | JSON list of `{label, bbox, score}` dicts |
| `structuredMarkdown` | string | reading-order-recovered Markdown |
| `tableHtmlJson` | string | JSON list of `{region_id, html}` |
| `pageType` | string | `'typeset'` / `'manuscript_cursive'` / `'image_only'` |

**Inputs**

- `notebooks/_artifacts/03b_fusion/fusion.json`
- PAGEs in Neo4j with `fusionStatus IN ['ok','single']`.
- Preprocessed images in MinIO `ancient-pages` bucket.

**Outputs**

- `notebooks/_artifacts/04_layout_analysis/report.json`
- PAGEs gain all layout properties listed above.

**Next**: Phase 5 — `05_chunking.ipynb` (semantic chunking using
`structuredMarkdown` for heading/table boundaries + `textFused` fallback).

In [ ]:
from __future__ import annotations

import json
import logging
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'pyproject.toml').exists(), f'cannot locate repo root from {Path.cwd()}'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loaded = load_dotenv(REPO_ROOT / '.env')
logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logging.getLogger('neo4j.notifications').setLevel(logging.WARNING)
logging.getLogger('apps.backend.pipeline.layout').setLevel(logging.INFO)

ARTIFACT_DIR = REPO_ROOT / 'notebooks' / '_artifacts' / '04_layout_analysis'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PRIOR = REPO_ROOT / 'notebooks' / '_artifacts' / '03b_fusion' / 'fusion.json'

RUN_FULL = os.getenv('RUN_FULL', '0') == '1'
MAX_PAGES = int(os.getenv('MAX_PAGES', '0' if RUN_FULL else '20'))
MAX_PAGES = None if RUN_FULL else MAX_PAGES
RECOMPUTE = os.getenv('RECOMPUTE', '0') == '1'
DOCUMENT_ID = os.getenv('DOCUMENT_ID') or None

print(f'repo root    : {REPO_ROOT}')
print(f'.env loaded  : {loaded}')
print(f'artifact dir : {ARTIFACT_DIR}')
print(f'RUN_FULL     : {RUN_FULL}  (MAX_PAGES={MAX_PAGES})')
print(f'RECOMPUTE    : {RECOMPUTE}')
print(f'DOCUMENT_ID  : {DOCUMENT_ID!r}')

repo root    : /Users/mohasani/Ancient
.env loaded  : True
artifact dir : /Users/mohasani/Ancient/notebooks/_artifacts/04_layout_analysis
RUN_FULL     : False  (MAX_PAGES=20)
RECOMPUTE    : False
DOCUMENT_ID  : None


In [ ]:
if not PRIOR.exists():
    raise RuntimeError(
        f'Missing prior artifact: {PRIOR}. Run 03b_fusion.ipynb first.'
    )
prior_data = json.loads(PRIOR.read_text())
print('Phase 3b status (from prior artifact):')
print(f'  pages_fused   : {prior_data.get("pages_fused")}')
print(f'  pages_failed  : {prior_data.get("pages_failed")}')
print(f'  agreement_mean: {prior_data.get("agreement_mean")}')

Phase 3b status (from prior artifact):
  pages_fused   : None
  pages_failed  : None
  agreement_mean: None


In [ ]:
from apps.backend.graph.neo4j_client import get_driver
from apps.backend.graph.schema import init_schema
from apps.backend.storage.minio_client import get_minio_client

driver = get_driver()
minio_client = get_minio_client()

# Apply new layout indexes idempotently.
schema_report = init_schema(driver)
new_indexes = [i for i in schema_report['lookup_indexes']
               if i['name'] in ('page_layout_status_index', 'page_type_index')]
print('Layout indexes:')
for idx in new_indexes:
    print(f'  {idx["name"]:30s}  state={idx["state"]}')

# Corpus snapshot.
with driver.session() as session:
    snap = session.run('''
        MATCH (p:PAGE)
        WHERE p.mode = 'ocr'
        RETURN
          count(p) AS total_ocr_pages,
          count(CASE WHEN p.fusionStatus IN ['ok','single'] THEN 1 END) AS eligible_for_layout,
          count(CASE WHEN p.layoutStatus IS NOT NULL THEN 1 END) AS already_have_layout,
          count(CASE WHEN p.layoutStatus = 'ok' THEN 1 END) AS layout_ok,
          count(CASE WHEN p.layoutStatus = 'manuscript' THEN 1 END) AS layout_manuscript
    ''').single()
print('\nCorpus snapshot:')
for k, v in dict(snap).items():
    print(f'  {k:>22}: {v}')

Layout indexes:
  page_layout_status_index        state=ONLINE
  page_type_index                 state=ONLINE

Corpus snapshot:
         total_ocr_pages: 7476
     eligible_for_layout: 60
     already_have_layout: 2301
               layout_ok: 2259
       layout_manuscript: 0


## 1. Smoke demo — single known-good page

Run `StructureEngine.analyse_page` on `毛汉光_中国中古社会史论::p00190`
(highest-confidence PaddleOCR page from Phase 3) and display the detected
regions and resulting Markdown. This triggers the PP-DocLayout_plus-L
model download (~126 MB) on first run.

In [ ]:
import cv2
import numpy as np
from IPython.display import Markdown, display

from apps.backend.ocr.structure import StructureEngine
from apps.backend.pipeline.layout import _uri_to_bucket_key

# Fetch the preprocessed image URI for the demo page.
_DEMO_PAGE_PREFIX = '毛汉光_中国中古社会史论__2d9f35b8db::p00190'

with driver.session() as session:
    demo_row = session.run(
        'MATCH (p:PAGE) WHERE p.id STARTS WITH $prefix RETURN p.id AS id, p.preprocessedImageUri AS uri LIMIT 1',
        prefix=_DEMO_PAGE_PREFIX,
    ).single()

if demo_row is None:
    print(f'Demo page not found (prefix={_DEMO_PAGE_PREFIX!r}). Trying any high-confidence page...')
    with driver.session() as session:
        demo_row = session.run(
            '''
            MATCH (p:PAGE)
            WHERE p.mode='ocr'
              AND p.fusionStatus IN ['ok', 'single']
              AND p.fusionCharCount > 100
              AND p.preprocessedImageUri IS NOT NULL
            RETURN p.id AS id, p.preprocessedImageUri AS uri
            ORDER BY p.paddleOcrConfidence DESC LIMIT 1
            '''
        ).single()

assert demo_row is not None, 'No eligible demo page found — run 03_dual_extraction + 03b_fusion first.'
demo_page_id = demo_row['id']
demo_uri = demo_row['uri']
print(f'Demo page: {demo_page_id}')
print(f'Image URI: {demo_uri}')

# Download image — URIs are bare keys (<doc_id>/page_N/final.png); bucket is implicit.
_bucket, _key = _uri_to_bucket_key(demo_uri)
print(f'MinIO: bucket={_bucket!r} key={_key!r}')
resp = minio_client.get_object(_bucket, _key)
try:
    _img_bytes = resp.read()
finally:
    resp.close()
    resp.release_conn()
_arr = np.frombuffer(_img_bytes, dtype=np.uint8)
demo_image = cv2.imdecode(_arr, cv2.IMREAD_COLOR)
print(f'Image shape: {demo_image.shape}')

# Run layout analysis.
demo_engine = StructureEngine()
demo_result = demo_engine.analyse_page(demo_image, page_id=demo_page_id)

print(f'\nLayout result:')
print(f'  error         : {demo_result.error}')
print(f'  region_count  : {demo_result.region_count}')
print(f'  duration_s    : {demo_result.duration_seconds:.2f}s')
print(f'  table_count   : {len(demo_result.table_html_list)}')
print(f'  markdown_chars: {len(demo_result.markdown)}')

Demo page: 毛汉光_中国中古社会史论__2d9f35b8db::p00190
Image URI: 毛汉光_中国中古社会史论__2d9f35b8db/page_00190/final.png


ValueError: invalid bucket name 毛汉光_中国中古社会史论__2d9f35b8db

In [ ]:
# Display detected regions as a table.
if demo_result.regions:
    print(f'{'Label':>20}  {'Score':>6}  {'Bbox (x,y,w,h)':>22}  Text preview')
    print('-' * 80)
    for r in demo_result.regions[:25]:
        bbox_str = str(r.bbox) if r.bbox else 'None'
        preview = (r.text or '')[:30].replace('\n', '↵')
        print(f'{r.label:>20}  {r.score:>6.3f}  {bbox_str:>22}  {preview}')
    if len(demo_result.regions) > 25:
        print(f'  ... ({len(demo_result.regions) - 25} more regions)')
else:
    print('No regions detected (empty result).')

In [ ]:
# Render the structured Markdown.
if demo_result.markdown:
    print('=== structuredMarkdown (first 1500 chars) ===')
    print(demo_result.markdown[:1500])
else:
    print('No markdown output.')

## 2. Full corpus run

Set `RUN_FULL=1` (env var) or change the cell below to run the full corpus.
For the background multi-hour run use:

```bash
caffeinate -dimsu uv run python scripts/run_layout_analysis.py --verbose
```

In [ ]:
from apps.backend.pipeline.layout import LayoutRunReport, run_layout_pages

if not RUN_FULL and MAX_PAGES == 0:
    print('Smoke run skipped (MAX_PAGES=0). Set RUN_FULL=1 or MAX_PAGES>0 to run.')
    layout_report = LayoutRunReport()  # empty report for artifact cell
else:
    print(f'Running layout analysis on {MAX_PAGES or "all"} pages...')
    layout_report = run_layout_pages(
        driver=driver,
        minio_client=minio_client,
        engine=demo_engine,  # reuse the already-loaded model
        recompute_existing=RECOMPUTE,
        document_id=DOCUMENT_ID,
        max_pages=MAX_PAGES,
        progress_every=10,
    )
    print(layout_report)

## 3. Artifact

In [ ]:
artifact = {
    'stage': '04_layout_analysis',
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'run_full': RUN_FULL,
    'max_pages': MAX_PAGES,
    'pages_total': layout_report.pages_total,
    'pages_ok': layout_report.pages_ok,
    'pages_empty': layout_report.pages_empty,
    'pages_failed': layout_report.pages_failed,
    'pages_manuscript': layout_report.pages_manuscript,
    'total_duration_seconds': layout_report.total_duration_seconds,
    'avg_seconds_per_page': layout_report.avg_seconds_per_page,
    'errors': layout_report.errors[:50],
    'sample_outcomes': layout_report.sample_outcomes[:10],
    'smoke_demo': {
        'page_id': demo_page_id,
        'region_count': demo_result.region_count,
        'duration_seconds': demo_result.duration_seconds,
        'markdown_chars': len(demo_result.markdown),
        'error': demo_result.error,
    },
}

artifact_path = ARTIFACT_DIR / 'report.json'
artifact_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f'Artifact written: {artifact_path}')
assert artifact_path.exists()

print(f'\nSummary:')
print(f'  total       : {artifact["pages_total"]}')
print(f'  ok          : {artifact["pages_ok"]}')
print(f'  empty       : {artifact["pages_empty"]}')
print(f'  failed      : {artifact["pages_failed"]}')
print(f'  manuscript  : {artifact["pages_manuscript"]}')
print(f'  duration    : {artifact["total_duration_seconds"]/60:.1f} min')
print(f'  avg/page    : {artifact["avg_seconds_per_page"]:.2f}s')

## 4. Verification

In [ ]:
# Layout status breakdown.
with driver.session() as session:
    status_rows = session.run('''
        MATCH (p:PAGE)
        WHERE p.layoutStatus IS NOT NULL
        RETURN p.layoutStatus AS status, count(*) AS n
        ORDER BY n DESC
    ''').data()

print('Layout status breakdown:')
for row in status_rows:
    print(f'  {row["status"]:>12}: {row["n"]}')

# Page type breakdown.
with driver.session() as session:
    type_rows = session.run('''
        MATCH (p:PAGE)
        WHERE p.pageType IS NOT NULL
        RETURN p.pageType AS page_type, count(*) AS n
        ORDER BY n DESC
    ''').data()

print('\nPage type breakdown:')
for row in type_rows:
    print(f'  {row["page_type"]:>20}: {row["n"]}')

In [ ]:
# Show 3 example structuredMarkdown snippets from high-region-count pages.
with driver.session() as session:
    examples = session.run('''
        MATCH (p:PAGE)
        WHERE p.layoutStatus = 'ok' AND p.layoutRegionCount >= 5
        RETURN p.id AS page_id,
               p.layoutRegionCount AS regions,
               p.structuredMarkdown AS md
        ORDER BY p.layoutRegionCount DESC
        LIMIT 3
    ''').data()

from IPython.display import Markdown, display
for ex in examples:
    print(f'--- {ex["page_id"]} ({ex["regions"]} regions) ---')
    md_preview = (ex.get('md') or '')[:800]
    if md_preview:
        display(Markdown(md_preview))
    else:
        print('  (no markdown)')
    print()

## 5. Manuscript page summary

Which documents are predominantly handwritten? These will rely on `textFused`
in Phase 5 and on specialised HTR tools (Kraken/Transkribus) in Phase 8.

In [ ]:
with driver.session() as session:
    ms_rows = session.run('''
        MATCH (p:PAGE {pageType: 'manuscript_cursive'})
        RETURN p.documentId AS document_id, count(*) AS manuscript_pages
        ORDER BY manuscript_pages DESC
        LIMIT 20
    ''').data()

if ms_rows:
    print(f'Documents with manuscript_cursive pages (top {len(ms_rows)}):')
    for row in ms_rows:
        print(f'  {row["manuscript_pages"]:>4} pages  {row["document_id"]}')
else:
    print('No manuscript_cursive pages classified yet (run corpus first).')

# HITL override example (commented out — uncomment to retag a specific page):
# from apps.backend.pipeline.layout import tag_manuscript_page
# tag_manuscript_page(driver, '唐耕耦、陆宏基_敦煌社会经济文献真迹释录_第二辑__9c6014dd26::p00459')